In [ ]:
!pip install wikipedia-api
from pydantic import BaseModel
import wikipediaapi
from typing import List, Optional

class InstitutionDetails(BaseModel):
    founder: Optional[str]
    founded: Optional[str]
    branches: Optional[List[str]]
    number_of_employees: Optional[int]
    summary: Optional[str]

def fetch_institution_details(name: str) -> InstitutionDetails:
    wiki_wiki = wikipediaapi.Wikipedia(user_agent='your-user-agent', language='en')
    page = wiki_wiki.page(name)
    if not page.exists():
        raise ValueError(f"'{name}' does not exist on Wikipedia.")
    info = {line.split(':')[0].strip(): line.split(':')[-1].strip() for line in page.text.split('\n') if ':' in line}
    return InstitutionDetails(
        founder=info.get('Founder', None),
        founded=info.get('Founded', None),
        branches=info.get('Branches', '').split(',') if 'Branches' in info else None,
        number_of_employees=int(info.get('Number of employees', '0').replace(',', '') or 0),
        summary=page.summary[:500]
    )

def display_institution_details(details: InstitutionDetails):
    print(
        f"Founder: {details.founder or 'N/A'}\n"
        f"Founded: {details.founded or 'N/A'}\n"
        f"Branches: {', '.join(details.branches) if details.branches else 'N/A'}\n"
        f"Employees: {details.number_of_employees or 'N/A'}\n"
        f"Summary: {details.summary or 'N/A'}"
    )

def main():
    institution_name = input("Enter institution name: ").strip()
    if institution_name:
        try:
            details = fetch_institution_details(institution_name)
            display_institution_details(details)
        except ValueError as e:
            print(e)
    else:
        print(" ❌ Please enter a valid institution name.")

if __name__ == "__main__":
    main()